# ENGE707 — Final Facial Feature Classification

**Selected Kaggle public score: 0.9753 (97.53%)**

This is the cleaned **final Version 2.1** pipeline: a fully fine-tuned ConvNeXt-Base combined with previously trained DINOv2-Large (LoRA) predictions at validation-selected weights of **50% / 50%**.

This notebook contains the final data preparation, training, prediction and ensemble methods. Historical experiments, manually reviewed test-image lists, OOD experiments, and exploratory submissions are excluded.

**Important:**
- The final, valid `submission_v2_ensemble.csv` is included. You do **not** need to retrain to submit it.
- The `train/` and `test/` directories in the downloadable ZIP are empty placeholders: copy the original images from your PC into them.
- Expensive training/inference switches are **False** by default.
- Reproducing the **exact** final predictions requires the existing ConvNeXt checkpoint and saved DINOv2 probability files from the original `Kaggle/outputs/` workspace. Those large intermediate artifacts are deliberately not packaged.
- Optional training and inference cache files, if generated, go outside this clean repository into the original workspace or a sibling model cache.

Execute the configuration cell and then the **Final submission verification** cell for a quick, no-training check.


In [ ]:
# 1 — Configuration, imports and paths. Existing CUDA torch and project .venv are retained.
from pathlib import Path
import os, gc, re, json, time, random, hashlib, math, warnings
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image, ImageFilter
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'train_labels.csv').is_file() else HERE.parent
assert (ROOT / 'train_labels.csv').is_file(), 'Open this notebook in the final repository folder.'

# Historical checkpoints / probabilities are deliberately NOT bundled.
# When the old Kaggle project exists, keep using its trained artifacts.
LEGACY_ROOT = ROOT if (ROOT / 'outputs').is_dir() else ROOT.parent / 'Kaggle'
ARTIFACT_ROOT = (LEGACY_ROOT / 'outputs') if (LEGACY_ROOT / 'outputs').is_dir() else (ROOT.parent / 'ENGE707_Model_Cache')
OUT = ARTIFACT_ROOT / 'kaggle_v2'
OUT.mkdir(parents=True, exist_ok=True)

LABELS_CSV=ROOT/'train_labels.csv'
SAMPLE_CSV=ROOT/'sample_submission.csv'
assert SAMPLE_CSV.is_file()

RUN_CONVNEXT_TRAIN=False       # Set True after the smoke test
# DINOv2 is permanently disabled for training and inference in this notebook.
RUN_INFERENCE=False           # Generate ConvNeXt ONLY val/test probability files
RUN_IMPORT_OLD_DINO=False     # Copy original DINOv2 val/test probability files into v2 output
RUN_ENSEMBLE=False            # Choose weights on labelled validation ONLY
RUN_TARGETED_RECHECK=False    # Recheck max 500 suspicious test images
RUN_SMALL_SMOKE_TEST=False    # Quick one-batch forward/backward check (downloads ConvNeXt-Base)

NUM_WORKERS=0                 # Reliable in Windows notebook
INPUT_SIZES={'convnext':(256,320)}  # DINOv2 is read from existing saved probabilities
BATCH_SIZE={'convnext':1}
ACCUM_STEPS={'convnext':8}
EPOCHS={'convnext':10}
HEAD_WARMUP_FRACTION=0.15  # Short head-only warmup; full fine-tuning starts next epoch
MAX_HOURS={'convnext':12}
PATIENCE={'convnext':3}
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
CLASS_ORDER=['CatEye','CatFace','CatMouth','CatNose','DogEye','DogFace','DogMouth','DogNose']
CLASS_ORDER=sorted(CLASS_ORDER)  # Consistent with v1 checkpoints and Kaggle labels.
PROB_COLS=['p_'+c for c in CLASS_ORDER]
# Final submission is already included; model training/inference stays off by default.
sample = pd.read_csv(SAMPLE_CSV)
sample['image_id'] = sample.image_id.astype(int)
assert sample.image_id.is_unique and len(sample) == 29443

print('Root:',ROOT,'| Device:',DEVICE,'| optional external model cache:',OUT)
if DEVICE.type=='cuda':
    print(torch.cuda.get_device_name(0),'VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))

## 3 — Reproducible train/validation split, no image-ID leakage

In [ ]:
# 4 — Data indexing and safeguards (train/test ID namespaces intentionally separate).
EXT={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
def find_image_root(which):
    candidates=[ROOT/which/which,ROOT/which]
    for p in candidates:
        if p.is_dir() and any(f.suffix.lower() in EXT for f in p.iterdir() if f.is_file()):return p
    raise FileNotFoundError(f'Extract {which} images to {candidates[0]} or {candidates[1]}')

def index_images(folder):
    out={}
    for p in folder.rglob('*'):
        if p.is_file() and p.suffix.lower() in EXT:
            match=re.search(r'(\d+)$',p.stem)
            if match:
                k=int(match.group(1))
                if k in out:raise ValueError(f'Duplicate ID in {folder}: {k}')
                out[k]=p
    return out

# Training indexing postponed until actual training preparation; comparison-only works without images.
def prepare_data():
    train_paths=index_images(find_image_root('train'))
    labels=pd.read_csv(LABELS_CSV)
    assert {'image_id','class'}.issubset(labels.columns)
    labels.image_id=labels.image_id.astype(int)
    assert labels.image_id.is_unique and set(labels.image_id)==set(train_paths)
    assert sorted(labels['class'].unique().tolist())==CLASS_ORDER,'8-class mapping differs from expected.'
    labels['target']=labels['class'].map({c:i for i,c in enumerate(CLASS_ORDER)}).astype(int)
    labels['path']=labels.image_id.map(lambda k:str(train_paths[k]))
    group_file=OUT/'exact_groups.csv'
    if group_file.is_file():groups=pd.read_csv(group_file,dtype={'group_id':str})
    else:
        rows=[(k,hashlib.sha256(p.read_bytes()).hexdigest()) for k,p in train_paths.items()]
        groups=pd.DataFrame(rows,columns=['image_id','group_id'])
        groups.to_csv(group_file,index=False)
    groups.image_id=groups.image_id.astype(int)
    assert groups.image_id.is_unique and set(groups.image_id)==set(labels.image_id)
    labels=labels.merge(groups,on='image_id',validate='one_to_one')
    conflicts=labels.groupby('group_id')['class'].nunique()
    assert conflicts.max()==1,'Conflicting labels for byte-identical images; audit the training labels.'
    split_file=OUT/'train_val_split.csv'
    old_split=LEGACY_ROOT/'outputs'/'robust_recognition_pipeline'/'train_val_split.csv'
    if split_file.is_file() or old_split.is_file():
        src=split_file if split_file.is_file() else old_split
        spl=pd.read_csv(src)[['image_id','split']]
        spl.image_id=spl.image_id.astype(int)
        assert spl.image_id.is_unique and set(spl.image_id)==set(labels.image_id)
        labels=labels.merge(spl,on='image_id',validate='one_to_one')
    else:
        tr,va=next(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED).split(
            labels,labels.target,groups=labels.group_id))
        labels['split']='train';labels.loc[va,'split']='val'
    assert set(labels['split'])=={'train','val'}
    tr=labels[labels.split=='train'].copy().reset_index(drop=True)
    va=labels[labels.split=='val'].copy().reset_index(drop=True)
    assert set(tr.group_id).isdisjoint(va.group_id),'Duplicate leakage; regenerate group-aware split.'
    labels[['image_id','split']].to_csv(split_file,index=False)
    dim_path=OUT/'dimensions.csv'
    old_dim=LEGACY_ROOT/'outputs'/'robust_recognition_pipeline'/'original_image_dimensions.csv'
    if dim_path.exists():dim=pd.read_csv(dim_path)
    elif old_dim.exists():dim=pd.read_csv(old_dim)[['image_id','width','height']]
    else:
        rows=[]
        for k,p in train_paths.items():
            with Image.open(p) as im: w,h=im.size
            rows.append((k,w,h))
        dim=pd.DataFrame(rows,columns=['image_id','width','height'])
    dim.image_id=dim.image_id.astype(int)
    assert dim.image_id.is_unique and set(dim.image_id)==set(labels.image_id)
    dim.to_csv(dim_path,index=False)
    tr=tr.merge(dim,on='image_id',validate='one_to_one')
    va=va.merge(dim,on='image_id',validate='one_to_one')
    for t in (tr,va):
        t['log_size']=np.log(t.width*t.height)
        t['log_aspect']=np.log(t.width/t.height)
    size_edges=np.unique(np.quantile(tr.log_size,[.25,.50,.75]))
    aspect_edges=np.unique(np.quantile(tr.log_aspect,[.25,.50,.75]))
    for t in (tr,va):
        t['size_bin']=np.searchsorted(size_edges,t.log_size,side='right')
        t['aspect_bin']=np.searchsorted(aspect_edges,t.log_aspect,side='right')
    meta={'classes':CLASS_ORDER,'size_edges':size_edges.tolist(),
          'aspect_edges':aspect_edges.tolist(),'mean':list(MEAN),'std':list(STD)}
    (OUT/'meta.json').write_text(json.dumps(meta,indent=2))
    print('Prepared data — train:',len(tr),'validation:',len(va))
    return tr,va,meta

# This step hashes ~63,660 training images once; cached on later runs.
RUN_PREPARE=False
if RUN_PREPARE: train_df,val_df,meta=prepare_data()

## 4 — Aspect-preserving geometry and quality augmentation
Reflective square padding keeps the original feature visible. Mild framing, horizontal flips, rotations, lighting variation, blur, JPEG compression and resampling train robustness without destroying tiny images.

In [ ]:
# 5 — Transforms, augmentation and datasets
class ReflectCanvas:
    def __init__(self,training=False):self.training=training
    def __call__(self,image):
        a=np.asarray(image.convert('RGB'))
        h,w=a.shape[:2]
        scale=random.uniform(1,1.22) if self.training else 1.
        side=max(max(h,w),round(max(h,w)*scale))
        dx,dy=side-w,side-h
        left=random.randint(0,dx) if self.training else dx//2
        top=random.randint(0,dy) if self.training else dy//2
        return Image.fromarray(np.pad(a,((top,dy-top),(left,dx-left),(0,0)),
                                       mode='reflect' if min(h,w)>1 else 'edge'))

class QualityJitter:
    def __call__(self,image):
        if min(image.size)>=32 and random.random()<.18:
            original=image.size
            f=random.uniform(.55,.9)
            small=(max(2,round(original[0]*f)),max(2,round(original[1]*f)))
            image=image.resize(small,Image.Resampling.BICUBIC).resize(original,Image.Resampling.BICUBIC)
        if random.random()<.1:
            image=image.filter(ImageFilter.GaussianBlur(random.uniform(.1,.7)))
        if random.random()<.15:
            import io
            buffer=io.BytesIO();image.save(buffer,format='JPEG',quality=random.randint(40,85))
            buffer.seek(0)
            with Image.open(buffer) as im:image=im.convert('RGB').copy()
        return image

def make_transform(size,training=False):
    ops=[ReflectCanvas(training)]
    if training:
        ops.extend([
            QualityJitter(),
            T.RandomHorizontalFlip(.5),
            T.RandomApply([T.RandomAffine(degrees=7,translate=(.025,.025),scale=(.92,1.08),
                                         interpolation=InterpolationMode.BICUBIC,fill=(124,116,104))],p=.5),
            T.RandomApply([T.ColorJitter(brightness=.12,contrast=.12,saturation=.1,hue=.01)],p=.5),
            T.RandomGrayscale(.03)
        ])
    return T.Compose(ops+[
        T.Resize((size,size),interpolation=InterpolationMode.BICUBIC,antialias=True),
        T.ToTensor(),T.Normalize(MEAN,STD)
    ])

class TrainViews(Dataset):
    def __init__(self,table,arch):
        self.table=table.reset_index(drop=True)
        self.tf=make_transform(INPUT_SIZES[arch][0],training=True)
    def __len__(self):return len(self.table)
    def __getitem__(self,i):
        row=self.table.iloc[i]
        with Image.open(row.path) as image:
            image=image.convert('RGB')
            a=self.tf(image);b=self.tf(image)
        return a,b,int(row.target),int(row.size_bin),int(row.aspect_bin)

class PredictDataset(Dataset):
    def __init__(self,ids,paths,size):
        self.ids=np.asarray(ids,dtype=int)
        self.paths=paths
        self.tf=make_transform(size,False)
    def __len__(self):return len(self.ids)
    def __getitem__(self,i):
        k=int(self.ids[i])
        with Image.open(self.paths[k]) as im:x=self.tf(im.convert('RGB'))
        return x,k

## 5 — Trainable ConvNeXt-Base and reused DINOv2 predictions

ConvNeXt-Base (~89M parameters) is **fully fine-tuned**. DINOv2-Large (~300M) is **already adapted with LoRA** from the previous run and is reused ONLY through its saved probability arrays. No DINOv2 model initialization, downloads, GPU inference or additional training occur in this notebook.


In [ ]:
# 6 — ConvNeXt-Base construction (there is deliberately NO DINOv2 model loader).
class GradientReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,scale):
        ctx.scale=float(scale)
        return x.view_as(x)
    @staticmethod
    def backward(ctx,gradient):
        return -ctx.scale*gradient,None

class ConvNeXtNet(nn.Module):
    def __init__(self,meta,pretrained=True):
        super().__init__()
        weights=models.ConvNeXt_Base_Weights.DEFAULT if pretrained else None
        self.backbone=models.convnext_base(weights=weights)
        width=self.backbone.classifier[-1].in_features
        self.backbone.classifier[-1]=nn.Identity()
        self.projector=nn.Identity()
        self.classifier=nn.Sequential(nn.Dropout(.15),nn.Linear(width,8))
        self.size_head=nn.Sequential(nn.Linear(width,128),nn.ReLU(),nn.Linear(128,len(meta['size_edges'])+1))
        self.aspect_head=nn.Sequential(nn.Linear(width,128),nn.ReLU(),nn.Linear(128,len(meta['aspect_edges'])+1))
    def forward(self,x,grl=0.,aux=False):
        z=self.projector(self.backbone(x)).float()
        logits=self.classifier(z)
        if aux:
            rev=GradientReverse.apply(z,grl)
            return logits,z,self.size_head(rev),self.aspect_head(rev)
        return logits,z

def make_model(arch,meta,pretrained=True):
    assert arch=='convnext', 'DINOv2 is reused via saved probabilities; never retrained here.'
    return ConvNeXtNet(meta,pretrained)

def model_folder(arch):
    p=OUT/arch
    p.mkdir(exist_ok=True)
    return p

def save_best_model(arch,net,meta,epoch,accuracy):
    assert arch=='convnext'
    contents={'meta':meta,'epoch':epoch,'val_accuracy':accuracy,'model':net.state_dict()}
    torch.save(contents,model_folder(arch)/'best.pt')

def load_best_model(arch,meta):
    assert arch=='convnext', 'Only ConvNeXt is loaded into GPU memory.'
    contents=torch.load(model_folder(arch)/'best.pt',map_location='cpu',weights_only=False)
    assert contents['meta']['classes']==meta['classes']
    net=make_model(arch,meta,pretrained=False)
    net.load_state_dict(contents['model'],strict=True)
    return net.to(DEVICE).eval()

def release_model(net):
    del net;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()


In [ ]:
# 7 — Optional one-batch smoke test (first do data preparation).
if RUN_SMALL_SMOKE_TEST:
    train_df,val_df,meta=prepare_data()
    assert DEVICE.type=='cuda','GPU recommended for either pretrained model.'
    smoke=make_model('convnext',meta,pretrained=True).to(DEVICE).train()
    a,b,y,s,asp=next(iter(DataLoader(TrainViews(train_df.head(2),'convnext'),batch_size=1)))
    with torch.amp.autocast('cuda'):
        logits,features,size_logits,asp_logits=smoke(a.to(DEVICE),grl=.05,aux=True)
        loss=(F.cross_entropy(logits.float(),y.to(DEVICE))+
              .1*F.cross_entropy(size_logits.float(),s.to(DEVICE))+
              .1*F.cross_entropy(asp_logits.float(),asp.to(DEVICE)))
    loss.backward()
    print('ConvNeXt-Base smoke test successful; loss:',float(loss.detach()))
    release_model(smoke)

## 6 — Fully fine-tune ConvNeXt-Base ONLY

ConvNeXt-Base trains with a SHORT 15% data head warmup, followed by full-network fine-tuning, two-view consistency, and a geometry-adversarial auxiliary loss. The best checkpoint and optimizer state are saved by epoch for resuming. **DINOv2 never trains.**


In [ ]:
# 8 — ConvNeXt-Base training with cached, resumable checkpoints.
@torch.inference_mode()
def val_evaluate(net,df,arch='convnext'):
    assert arch=='convnext'
    size=INPUT_SIZES['convnext'][0]
    paths={int(r.image_id):Path(r.path) for r in df.itertuples(index=False)}
    loader=DataLoader(PredictDataset(df.image_id,paths,size),
                      batch_size=4,shuffle=False,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda')
    truth=df.set_index('image_id')['target']
    ys=[];yp=[];net.eval()
    for x,ids in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
            p1=net(x)[0].float().softmax(-1)
            p2=net(torch.flip(x,dims=[3]))[0].float().softmax(-1)
        ps=(p1+p2)/2
        keys=ids.tolist()
        ys.extend(truth.loc[keys].tolist())
        yp.extend(ps.argmax(1).cpu().tolist())
    return accuracy_score(ys,yp),f1_score(ys,yp,average='macro')

def train_convnext_base():
    assert DEVICE.type=='cuda','ConvNeXt-Base full fine-tuning requires CUDA.'
    tr,va,meta=prepare_data()
    folder=model_folder('convnext')
    net=make_model('convnext',meta,pretrained=True).to(DEVICE)
    backbone=list(net.backbone.parameters())
    heads=[p for module in (net.projector,net.classifier,net.size_head,net.aspect_head)
           for p in module.parameters()]
    optimizer=torch.optim.AdamW([{'params':backbone,'lr':2e-5},
                                 {'params':heads,'lr':2e-4}],weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda')
    loader=DataLoader(TrainViews(tr,'convnext'),batch_size=BATCH_SIZE['convnext'],
                      shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
    warmup_rows=tr.sample(frac=HEAD_WARMUP_FRACTION,random_state=SEED)
    warmup_loader=DataLoader(TrainViews(warmup_rows,'convnext'),batch_size=BATCH_SIZE['convnext'],
                             shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
    latest=folder/'last_state.pt'
    first=0;best=-1.;stale=0;history=[]
    if latest.exists():
        state=torch.load(latest,map_location='cpu',weights_only=False)
        assert state['meta']['classes']==CLASS_ORDER
        net.load_state_dict(state['model'],strict=True)
        optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        first=state['next_epoch'];best=state['best'];stale=state['stale'];history=state['history']
        print('Resuming from epoch',first)
    started=time.monotonic()
    for epoch in range(first,EPOCHS['convnext']):
        warmup=(epoch==0)
        for p in backbone:p.requires_grad_(not warmup)
        net.train()
        if warmup:net.backbone.eval()
        grl=0. if warmup else .12*min(1.,epoch/4)
        consistency=0. if warmup else .20*min(1.,epoch/2)
        optimizer.zero_grad(set_to_none=True)
        epoch_loader=warmup_loader if warmup else loader
        ce_sum=0.;count=0
        for step,(a,b,y,sz,asp) in enumerate(epoch_loader,1):
            a=a.to(DEVICE);y=y.to(DEVICE);sz=sz.to(DEVICE);asp=asp.to(DEVICE)
            teacher=None
            if consistency:
                net.eval()
                with torch.no_grad(),torch.amp.autocast('cuda'):
                    teacher=net(b.to(DEVICE))[0].float().softmax(-1).detach()
                net.train()
            with torch.amp.autocast('cuda'):
                logits,z,ls,la=net(a,grl=grl,aux=True)
                ce=F.cross_entropy(logits.float(),y)
                geom=(F.cross_entropy(ls.float(),sz)+F.cross_entropy(la.float(),asp))/2
                kl=(F.kl_div(F.log_softmax(logits.float(),dim=-1),teacher,
                              reduction='batchmean') if teacher is not None else logits.sum()*0)
                loss=ce+.2*geom+consistency*kl
            scaler.scale(loss/ACCUM_STEPS['convnext']).backward()
            if step%ACCUM_STEPS['convnext']==0 or step==len(epoch_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(backbone+heads,1.)
                scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
            ce_sum+=float(ce.detach());count+=1
            if step%5000==0:print('ConvNeXt-Base',epoch+1,step,'/',len(epoch_loader))
        accuracy,macro_f1=val_evaluate(net,va)
        if accuracy>best+1e-6:
            best=accuracy;stale=0
            save_best_model('convnext',net,meta,epoch,accuracy)
        else:stale+=1
        history.append({'epoch':epoch+1,'train_ce':ce_sum/max(count,1),
                        'validation_accuracy':accuracy,'validation_f1':macro_f1,'best':best})
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        torch.save({'meta':meta,'next_epoch':epoch+1,'best':best,'stale':stale,
                    'history':history,'optimizer':optimizer.state_dict(),
                    'scaler':scaler.state_dict(),'model':net.state_dict()},latest)
        print(f'ConvNeXt-Base epoch {epoch+1}: accuracy {accuracy:.5f}; F1 {macro_f1:.5f}; best {best:.5f}')
        if stale>=PATIENCE['convnext']:
            print('Early stopping');break
        if (time.monotonic()-started)/3600>=MAX_HOURS['convnext']:
            print('Time cap reached after full epoch');break
    release_model(net)

if RUN_CONVNEXT_TRAIN:train_convnext_base()


## 7 — ConvNeXt-Base validation and test probabilities

Only ConvNeXt-Base runs inference on the GPU (256px and 320px, flip TTA). Both passes are cached by image ID. In the following cell, the already-trained DINOv2 validation and test probabilities are imported straight from your previous `outputs/dinov2_vitl14_lora` folder.


In [ ]:
# 9 — ConvNeXt-Base probabilities and feature vectors, saved by image ID.
@torch.inference_mode()
def inference_convnext(subset):
    assert subset in ('val','test')
    tr,va,meta=prepare_data()
    net=load_best_model('convnext',meta)
    if subset=='val':
        ids=va.image_id.astype(int).to_numpy()
        paths={int(r.image_id):Path(r.path) for r in va.itertuples(index=False)}
    else:
        index=index_images(find_image_root('test'))
        ids=sample.image_id.astype(int).to_numpy()
        assert set(ids).issubset(index)
        paths=index
    folder=model_folder('convnext')
    for size in INPUT_SIZES['convnext']:
        path=folder/f'{subset}_{size}_probabilities.csv'
        feature_path=folder/f'{subset}_{size}_features.npz'
        if path.exists() and feature_path.exists():
            df=pd.read_csv(path)
            if df.image_id.astype(int).tolist()==ids.tolist() and list(df.columns)==['image_id']+PROB_COLS:
                print('Using cached:',path.name)
                continue
            raise ValueError(f'Stale inference cache {path}; remove it before rerunning')
        loader=DataLoader(PredictDataset(ids,paths,size),batch_size=4,
                          num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
        probs=[];features=[];output_ids=[];net.eval()
        for step,(x,batch_ids) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                l1,f1=net(x);l2,f2=net(torch.flip(x,dims=[3]))
            probs.append(((l1.float().softmax(-1)+l2.float().softmax(-1))/2).cpu().numpy())
            features.append(((f1.float()+f2.float())/2).cpu().numpy())
            output_ids.extend(batch_ids.tolist())
            if step%3000==0:print('ConvNeXt',subset,size,step,'/',len(loader))
        assert output_ids==ids.tolist()
        df=pd.DataFrame(np.concatenate(probs),columns=PROB_COLS)
        df.insert(0,'image_id',output_ids)
        df.to_csv(path,index=False)
        np.savez_compressed(feature_path,image_id=np.asarray(output_ids),
                            features=np.concatenate(features).astype(np.float32))
        print('Saved:',path)
    release_model(net)

if RUN_INFERENCE:
    for subset in ('val','test'):inference_convnext(subset)


In [ ]:
# 9b — Import the OLD, already-trained DINOv2-Large predictions. No DINOv2 GPU use.
def import_old_dino_probabilities():
    tr,va,meta=prepare_data()
    old=LEGACY_ROOT/'outputs'/'dinov2_vitl14_lora'
    val_file=old/'best_validation_probabilities.npz'
    test_file=old/'test_probabilities.csv'
    for path in (val_file,test_file):
        assert path.is_file(),f'Missing existing DINOv2 predictions: {path}'

    val_ids=va.image_id.astype(int).to_numpy()
    test_ids=sample.image_id.astype(int).to_numpy()
    with np.load(val_file,allow_pickle=False) as obj:
        old_val_ids=np.asarray(obj['image_id']).astype(int)
        old_val_p=np.asarray(obj['probs'],dtype=np.float64)
        if 'classes' in obj.files:
            assert list(obj['classes'].astype(str))==CLASS_ORDER,'DINOv2 class order differs'
    assert old_val_p.shape==(len(old_val_ids),len(CLASS_ORDER))
    assert len(set(old_val_ids))==len(old_val_ids)
    assert set(old_val_ids)==set(val_ids),(
        'Your v2 validation split differs from the split used to train DINOv2. '
        'Restore outputs/robust_recognition_pipeline/train_val_split.csv before importing.'
    )
    old_val_p=old_val_p[pd.Series(np.arange(len(old_val_ids)),index=old_val_ids).loc[val_ids].to_numpy()]
    old_test=pd.read_csv(test_file)
    old_test.image_id=old_test.image_id.astype(int)
    assert old_test.image_id.is_unique and set(old_test.image_id)==set(test_ids)
    assert set(PROB_COLS).issubset(old_test.columns),('Missing class probability columns',PROB_COLS)
    old_test_p=old_test.set_index('image_id').loc[test_ids,PROB_COLS].to_numpy(dtype=np.float64)

    for subset,ids,probs in (('val',val_ids,old_val_p),('test',test_ids,old_test_p)):
        assert np.isfinite(probs).all() and np.allclose(probs.sum(1),1.,atol=2e-4)
        df=pd.DataFrame(probs,columns=PROB_COLS)
        df.insert(0,'image_id',ids)
        path=OUT/f'dinov2_reused_{subset}_probabilities.csv'
        df.to_csv(path,index=False)
        print('Imported',subset,len(df),'DINOv2 predictions →',path)
    print('Existing DINOv2 model retained; ZERO DINOv2 training/inference.')
    print('Note: existing DINOv2 validation=336px TTA, test=336/392px multiscale TTA.')

if RUN_IMPORT_OLD_DINO:import_old_dino_probabilities()


## 8 — Blend NEW ConvNeXt-Base with OLD DINOv2-Large

Load saved DINOv2 probabilities. Optimize only the new ConvNeXt high-resolution contribution and the two-model blending weight on the validation calibration half. Report performance on the held-out audit half. There is **no second DINOv2 model or inference**.


In [ ]:
# 10 — Combine new ConvNeXt outputs with previously trained DINOv2 probabilities.
def get_probabilities(arch,subset,size,ids):
    assert arch=='convnext'
    path=model_folder('convnext')/f'{subset}_{size}_probabilities.csv'
    assert path.is_file(),f'Missing CNN probabilities: {path}. Set RUN_INFERENCE=True.'
    df=pd.read_csv(path)
    assert df.image_id.is_unique and set(df.image_id.astype(int))==set(np.asarray(ids,dtype=int))
    assert list(df.columns)==['image_id']+PROB_COLS
    p=df.set_index('image_id').loc[np.asarray(ids,dtype=int),PROB_COLS].to_numpy()
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1.,atol=2e-4)
    return p

def reused_dino_probabilities(subset,ids):
    path=OUT/f'dinov2_reused_{subset}_probabilities.csv'
    assert path.is_file(),f'Missing {path}; set RUN_IMPORT_OLD_DINO=True before the ensemble.'
    df=pd.read_csv(path)
    assert df.image_id.is_unique and set(df.image_id.astype(int))==set(np.asarray(ids,dtype=int))
    assert list(df.columns)==['image_id']+PROB_COLS
    p=df.set_index('image_id').loc[np.asarray(ids,dtype=int),PROB_COLS].to_numpy()
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1.,atol=2e-4)
    return p

def tune_ensemble():
    tr,va,meta=prepare_data()
    truth=va.target.to_numpy()
    val_ids=va.image_id.astype(int).to_numpy()
    test_ids=sample.image_id.astype(int).to_numpy()
    cal,audit=train_test_split(np.arange(len(va)),stratify=truth,test_size=.5,random_state=SEED)
    low,high=INPUT_SIZES['convnext']
    cv_low=get_probabilities('convnext','val',low,val_ids)
    cv_high=get_probabilities('convnext','val',high,val_ids)
    ct_low=get_probabilities('convnext','test',low,test_ids)
    ct_high=get_probabilities('convnext','test',high,test_ids)
    dv=reused_dino_probabilities('val',val_ids)
    dt=reused_dino_probabilities('test',test_ids)
    trials=[]
    for hc in (0.,.25,.50):
        cv=(1-hc)*cv_low+hc*cv_high
        for wc in np.round(np.arange(0,1.001,.05),2):
            blend=wc*cv+(1-wc)*dv
            mistakes=int((blend[cal].argmax(1)!=truth[cal]).sum())
            trials.append((mistakes,abs(wc-.75),hc,wc))
    _,_,hc,wc=min(trials)
    cv=(1-hc)*cv_low+hc*cv_high
    ct=(1-hc)*ct_low+hc*ct_high
    val_blend=wc*cv+(1-wc)*dv
    test_blend=wc*ct+(1-wc)*dt
    print(f'Chosen new CNN weight: {wc:.2f}; old DINOv2 weight: {1-wc:.2f}; CNN high-res: {hc:.2f}')
    print('Caution: old DINOv2 val=336 TTA and test=336/392 multiscale TTA.')
    for label,p in [('ConvNeXt-Base',cv),('Existing DINOv2-Large',dv),('Combined',val_blend)]:
        print(label,'held-out validation accuracy:',f'{accuracy_score(truth[audit],p[audit].argmax(1)):.5%}',
              '| mistakes:',int((truth[audit]!=p[audit].argmax(1)).sum()))
    pd.DataFrame({'image_id':val_ids,'true_class':va['class'],
                  'predicted_class':[CLASS_ORDER[i] for i in val_blend.argmax(1)],
                  'confidence':val_blend.max(1)}).to_csv(OUT/'v2_validation_predictions.csv',index=False)
    def save_prob(path,ids,p):
        df=pd.DataFrame(p,columns=PROB_COLS);df.insert(0,'image_id',ids)
        df.to_csv(path,index=False)
    for name,ids,p in [
        ('ensemble_val_probs.csv',val_ids,val_blend),
        ('ensemble_test_probs.csv',test_ids,test_blend),
        ('convnext_val_blend_probs.csv',val_ids,cv),
        ('convnext_test_blend_probs.csv',test_ids,ct),
        ('dinov2_val_blend_probs.csv',val_ids,dv),
        ('dinov2_test_blend_probs.csv',test_ids,dt)
    ]:
        save_prob(OUT/name,ids,p)
    config={'convnext_weight':float(wc),'convnext_highres':float(hc),
            'dinov2_source':'reused original LoRA predictions',
            'dinov2_validation':'336px flip-TTA',
            'dinov2_test':'336/392px multiscale flip-TTA',
            'calibration_size':len(cal),'audit_size':len(audit)}
    (OUT/'ensemble_config.json').write_text(json.dumps(config,indent=2))
    submission=sample[['image_id']].copy()
    submission['class']=[CLASS_ORDER[i] for i in test_blend.argmax(1)]
    submission.to_csv(OUT/'submission_v2_ensemble.csv',index=False)
    print('Saved:',OUT/'submission_v2_ensemble.csv')

if RUN_ENSEMBLE:tune_ensemble()


In [ ]:
# Final submission verification — no training, old checkpoints, or images required.
submission = pd.read_csv(ROOT / 'submission_v2_ensemble.csv')
assert list(submission.columns) == ['image_id', 'class']
assert submission.image_id.is_unique and len(submission) == len(sample) == 29443
assert submission.image_id.astype(int).tolist() == sample.image_id.astype(int).tolist()
assert submission['class'].isin(CLASS_ORDER).all()
assert submission['class'].notna().all()
print('PASS — 29,443 predictions, correct test IDs and eight valid classes')
print('Selected Kaggle public score: 0.9753')
